# Tempreture Persona Transformer Training

This notebook trains a PyTorch Transformer classifier for temperature/HVAC persona prediction using `temperatureData.csv`. It mounts Google Drive, builds recent room temperature sequences, trains on the `ac_persona` label, and saves model artifacts back to Drive as a zip file.


## Optional GitHub Clone
This notebook is self-contained, but this cell also clones the project code so your Colab runtime follows the same GitHub + Drive workflow.


In [9]:
from google.colab import drive
from pathlib import Path
import os
import subprocess

# 1) Mount Google Drive for datasets and saved training outputs.
drive.mount('/content/drive')

# 2) Edit these two values for your GitHub repo.
GITHUB_REPO_URL = 'https://github.com/abdullahtapanci/VisualizationApp.git'
GITHUB_BRANCH = 'main'  # change if you use another branch

# 3) Drive folders. Datasets stay in Drive; code is cloned from GitHub.
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/VisualizationApp')
DRIVE_DATA_DIR = DRIVE_PROJECT_DIR / 'Data'
DRIVE_OUTPUT_ROOT = DRIVE_PROJECT_DIR
PROJECT_DIR = Path('/content/VisualizationApp')

if 'YOUR_USERNAME' in GITHUB_REPO_URL:
    raise ValueError('Edit GITHUB_REPO_URL to your real GitHub repository URL, then rerun this cell.')
if not DRIVE_DATA_DIR.exists():
    raise FileNotFoundError(f'Drive dataset folder was not found: {DRIVE_DATA_DIR}')

# 4) Clone or update code from GitHub.
if PROJECT_DIR.exists():
    print('Repository already exists. Pulling latest changes...')
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', GITHUB_BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', 'origin', GITHUB_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', GITHUB_BRANCH, GITHUB_REPO_URL, str(PROJECT_DIR)], check=True)

# 5) Link Drive datasets into the cloned repo so existing scripts read Data/*.csv.
LOCAL_DATA_DIR = PROJECT_DIR / 'Data'
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
for filename in ['PIRSensorData.csv', 'hotelReservationData.csv', 'lightningData.csv', 'temperatureData.csv', 'WheatherDataAntalya.csv']:
    src = DRIVE_DATA_DIR / filename
    dst = LOCAL_DATA_DIR / filename
    if src.exists():
        if dst.exists() or dst.is_symlink():
            dst.unlink()
        os.symlink(src, dst)
        print('linked', dst, '->', src)
    else:
        print('not found in Drive, skipping:', src)

DATA_DIR = LOCAL_DATA_DIR
print('PROJECT_DIR =', PROJECT_DIR)
print('DATA_DIR =', DATA_DIR)
print('DRIVE_OUTPUT_ROOT =', DRIVE_OUTPUT_ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repository already exists. Pulling latest changes...
linked /content/VisualizationApp/Data/PIRSensorData.csv -> /content/drive/MyDrive/VisualizationApp/Data/PIRSensorData.csv
linked /content/VisualizationApp/Data/hotelReservationData.csv -> /content/drive/MyDrive/VisualizationApp/Data/hotelReservationData.csv
linked /content/VisualizationApp/Data/lightningData.csv -> /content/drive/MyDrive/VisualizationApp/Data/lightningData.csv
linked /content/VisualizationApp/Data/temperatureData.csv -> /content/drive/MyDrive/VisualizationApp/Data/temperatureData.csv
linked /content/VisualizationApp/Data/WheatherDataAntalya.csv -> /content/drive/MyDrive/VisualizationApp/Data/WheatherDataAntalya.csv
PROJECT_DIR = /content/VisualizationApp
DATA_DIR = /content/VisualizationApp/Data
DRIVE_OUTPUT_ROOT = /content/drive/MyDrive/VisualizationApp


In [10]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive')
DRIVE_PROJECT_DIR = DRIVE_ROOT / 'VisualizationApp'
DRIVE_DATA_DIR = DRIVE_PROJECT_DIR / 'Data'

# Prefer the dataset symlink created by the GitHub/Drive setup cell, then the
# actual Drive folder used by this project.
CANDIDATE_DATA_PATHS = []
if 'DATA_DIR' in globals():
    CANDIDATE_DATA_PATHS.append(Path(DATA_DIR) / 'temperatureData.csv')
CANDIDATE_DATA_PATHS.extend([
    DRIVE_PROJECT_DIR / 'Data' / 'temperatureData.csv',
    DRIVE_PROJECT_DIR / 'temperatureData.csv',
    DRIVE_ROOT / 'Data' / 'temperatureData.csv',
    DRIVE_ROOT / 'temperatureData.csv',
])

DATA_CSV = next((p for p in CANDIDATE_DATA_PATHS if p.exists()), CANDIDATE_DATA_PATHS[0])
OUT_DIR = DRIVE_PROJECT_DIR / 'AIModelsAndAlgorithms' / 'TempreturePersona' / 'transformer'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Candidate data paths:')
for candidate in CANDIDATE_DATA_PATHS:
    print(' -', candidate, 'exists:', candidate.exists())
print('DATA_CSV:', DATA_CSV)
print('DATA_CSV exists:', DATA_CSV.exists())
print('OUT_DIR:', OUT_DIR)

if not DATA_CSV.exists():
    raise FileNotFoundError(
        'Could not find temperatureData.csv. Expected it at '
        f'{DRIVE_PROJECT_DIR / "Data" / "temperatureData.csv"}. '
        'If your file is elsewhere, set DATA_CSV manually to that path and rerun this cell.'
    )


Candidate data paths:
 - /content/VisualizationApp/Data/temperatureData.csv exists: True
 - /content/drive/MyDrive/VisualizationApp/Data/temperatureData.csv exists: True
 - /content/drive/MyDrive/VisualizationApp/temperatureData.csv exists: False
 - /content/drive/MyDrive/Data/temperatureData.csv exists: False
 - /content/drive/MyDrive/temperatureData.csv exists: False
DATA_CSV: /content/VisualizationApp/Data/temperatureData.csv
DATA_CSV exists: True
OUT_DIR: /content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/TempreturePersona/transformer


In [12]:
import json
import math
import random
import zipfile

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)


device: cuda


In [13]:
# Training controls. Start with these safe defaults, then increase gradually.
LABEL_COL = 'ac_persona'
MAX_ROWS = 2_000_000       # Use less, e.g. 500_000, if the kernel runs out of RAM.
MAX_SEQUENCES = 300_000    # Sequence tensors are memory-heavy; reduce to 75_000 if needed.
SEQ_LEN = 24               # 24 samples = 2 hours if data is every 5 minutes.
STRIDE = 3                 # Larger stride uses fewer sequences and much less RAM.
BATCH_SIZE = 256           # Lower batch size reduces GPU/RAM pressure.
EPOCHS = 20
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
D_MODEL = 96
N_HEADS = 4
N_LAYERS = 2
DIM_FEEDFORWARD = 192
DROPOUT = 0.15
PATIENCE = 3

# Keep this True unless you are sure Colab has enough RAM for the full multi-GB dataset.
REQUIRE_ROW_LIMIT = True

# Most datasets use None for no meaningful AC persona. Keep this default if you want the model
# to learn only real temperature persona classes. Set to an empty set to include every label.
EXCLUDED_LABELS = {'None', 'Unknown', 'nan', ''}


In [14]:
USECOLS = [
    'timestamp', 'room_number', 'floor', 'facade', 'room_type', 'size_m2',
    'outside_temp', 'room_temp', 'setpoint', 'ideal_temp', 'hvac_mode',
    'ac_persona', 'occupant_state', 'pir_persona', 'room_state', 'pir_motion', 'guest_id',
]

BASE_NUMERIC_COLS = [
    'floor_scaled', 'size_scaled', 'outside_temp_scaled', 'room_temp_scaled',
    'setpoint_scaled', 'ideal_temp_scaled', 'temp_error_scaled', 'comfort_error_scaled',
    'pir_motion', 'has_guest', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
]
CAT_COLS = ['facade', 'room_type', 'hvac_mode', 'occupant_state', 'pir_persona', 'room_state']

def cyclic(values, period):
    radians = 2 * np.pi * values / period
    return np.sin(radians), np.cos(radians)

def scale_temp(series):
    return ((pd.to_numeric(series, errors='coerce').fillna(0).clip(-20, 50) + 20) / 70).astype('float32')

def load_and_prepare(data_csv, max_rows=None):
    if REQUIRE_ROW_LIMIT and max_rows is None:
        raise ValueError(
            'MAX_ROWS=None will try to load the full multi-GB temperature dataset and may crash the kernel. '
            'Increase MAX_ROWS gradually, or set REQUIRE_ROW_LIMIT=False only on a High-RAM runtime.'
        )
    print(f'Loading rows from {data_csv} with MAX_ROWS={max_rows} ...')
    df = pd.read_csv(data_csv, usecols=USECOLS, parse_dates=['timestamp'], nrows=max_rows)
    print('raw loaded rows:', len(df), 'approx memory MB:', round(df.memory_usage(deep=True).sum() / 1_000_000, 1))
    if LABEL_COL not in df.columns:
        raise ValueError(f'LABEL_COL={LABEL_COL!r} was not found. Available columns: {list(df.columns)}')

    df = df.dropna(subset=['timestamp']).copy()
    df = df.sort_values(['room_number', 'timestamp']).reset_index(drop=True)

    df[LABEL_COL] = df[LABEL_COL].fillna('Unknown').astype(str)
    if EXCLUDED_LABELS:
        df = df[~df[LABEL_COL].isin(EXCLUDED_LABELS)].copy()
    if df.empty:
        raise ValueError('No labeled rows left after filtering EXCLUDED_LABELS. Adjust EXCLUDED_LABELS.')
    df = df.sort_values(['room_number', 'timestamp']).reset_index(drop=True)

    for col in ['floor', 'size_m2', 'outside_temp', 'room_temp', 'setpoint', 'ideal_temp', 'pir_motion']:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    df['guest_id'] = pd.to_numeric(df['guest_id'], errors='coerce')

    for col in CAT_COLS:
        df[col] = df[col].fillna('Unknown').astype(str)

    df['floor_scaled'] = df['floor'].clip(0, 30) / 30.0
    df['size_scaled'] = df['size_m2'].clip(0, 120) / 120.0
    df['outside_temp_scaled'] = scale_temp(df['outside_temp'])
    df['room_temp_scaled'] = scale_temp(df['room_temp'])
    df['setpoint_scaled'] = scale_temp(df['setpoint'])
    df['ideal_temp_scaled'] = scale_temp(df['ideal_temp'])
    df['temp_error_scaled'] = ((df['room_temp'] - df['setpoint']).clip(-20, 20) + 20) / 40.0
    df['comfort_error_scaled'] = ((df['room_temp'] - df['ideal_temp']).clip(-20, 20) + 20) / 40.0
    df['pir_motion'] = df['pir_motion'].clip(0, 1).astype('float32')
    df['has_guest'] = df['guest_id'].notna().astype('float32')
    df['hour_sin'], df['hour_cos'] = cyclic(df['timestamp'].dt.hour, 24)
    df['dow_sin'], df['dow_cos'] = cyclic(df['timestamp'].dt.dayofweek, 7)

    for col in BASE_NUMERIC_COLS:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype('float32')
    return df

df = load_and_prepare(DATA_CSV, MAX_ROWS)
print('prepared rows:', len(df))
print('prepared memory MB:', round(df.memory_usage(deep=True).sum() / 1_000_000, 1))
print('time range:', df['timestamp'].min(), 'to', df['timestamp'].max())
print('labels:')
print(df[LABEL_COL].value_counts())
df.head()


Loading rows from /content/VisualizationApp/Data/temperatureData.csv with MAX_ROWS=2000000 ...
raw loaded rows: 2000000 approx memory MB: 852.2
prepared rows: 383661
prepared memory MB: 199.4
time range: 2022-01-06 19:05:00 to 2022-03-16 10:35:00
labels:
ac_persona
Reactive           137715
EnergySaver        120727
AlwaysOnComfort    113588
Preconditioning      7661
Housekeeping         3970
Name: count, dtype: int64


,timestamp,room_number,floor,facade,room_type,size_m2,outside_temp,room_temp,setpoint,ideal_temp,...,room_temp_scaled,setpoint_scaled,ideal_temp_scaled,temp_error_scaled,comfort_error_scaled,has_guest,hour_sin,hour_cos,dow_sin,dow_cos
0,2022-01-22 07:20:00,1,1,East,Deluxe,35,11.58,14.45,20.0,22.0,...,0.492143,0.571429,0.6,0.36125,0.31125,0.0,0.965926,-0.258819,-0.974928,-0.222521
1,2022-01-22 07:25:00,1,1,East,Deluxe,35,11.60,14.77,20.0,22.0,...,0.496714,0.571429,0.6,0.36925,0.31925,0.0,0.965926,-0.258819,-0.974928,-0.222521
2,2022-01-22 07:30:00,1,1,East,Deluxe,35,11.62,15.11,20.0,22.0,...,0.501571,0.571429,0.6,0.37775,0.32775,0.0,0.965926,-0.258819,-0.974928,-0.222521
3,2022-01-22 07:35:00,1,1,East,Deluxe,35,11.64,15.47,20.0,22.0,...,0.506714,0.571429,0.6,0.38675,0.33675,0.0,0.965926,-0.258819,-0.974928,-0.222521
4,2022-01-22 07:40:00,1,1,East,Deluxe,35,11.66,15.81,20.0,22.0,...,0.511571,0.571429,0.6,0.39525,0.34525,0.0,0.965926,-0.258819,-0.974928,-0.222521


In [15]:
classes = sorted(df[LABEL_COL].unique().tolist())
class_to_id = {label: i for i, label in enumerate(classes)}
df['label_id'] = df[LABEL_COL].map(class_to_id).astype('int64')

category_values = {}
for col in CAT_COLS:
    values = sorted(df[col].fillna('Unknown').astype(str).unique().tolist())
    if 'Unknown' not in values:
        values.append('Unknown')
    category_values[col] = values

feature_names = list(BASE_NUMERIC_COLS)
for col, values in category_values.items():
    feature_names.extend([f'{col}={value}' for value in values])

def encode_features(frame):
    parts = [frame[BASE_NUMERIC_COLS].to_numpy(dtype='float32')]
    for col, values in category_values.items():
        mapping = {value: i for i, value in enumerate(values)}
        unknown_idx = mapping.get('Unknown', 0)
        ids = frame[col].fillna('Unknown').astype(str).map(mapping).fillna(unknown_idx).astype(int).to_numpy()
        onehot = np.zeros((len(frame), len(values)), dtype='float32')
        onehot[np.arange(len(frame)), ids] = 1.0
        parts.append(onehot)
    return np.concatenate(parts, axis=1)

encoded = encode_features(df)
labels = df['label_id'].to_numpy(dtype='int64')
print('classes:', classes)
print('feature count:', encoded.shape[1])
print('categories:', {k: len(v) for k, v in category_values.items()})


classes: ['AlwaysOnComfort', 'EnergySaver', 'Housekeeping', 'Preconditioning', 'Reactive']
feature count: 42
categories: {'facade': 5, 'room_type': 4, 'hvac_mode': 4, 'occupant_state': 6, 'pir_persona': 5, 'room_state': 4}


In [16]:
def build_sequences(frame, encoded_features, labels, seq_len, stride=1, max_sequences=None):
    sequences = []
    y = []
    end_indices = []

    frame = frame.reset_index(drop=True)
    for _, group in frame.groupby('room_number', sort=False):
        idx = group.index.to_numpy(dtype=np.int64)
        if len(idx) < seq_len:
            continue
        for end in range(seq_len - 1, len(idx), stride):
            window_idx = idx[end - seq_len + 1:end + 1]
            sequences.append(encoded_features[window_idx])
            y.append(labels[idx[end]])
            end_indices.append(idx[end])
            if max_sequences and len(sequences) >= max_sequences:
                return (
                    np.stack(sequences).astype('float32'),
                    np.array(y, dtype='int64'),
                    np.array(end_indices, dtype=np.int64),
                )

    if not sequences:
        raise ValueError('No sequences were built. Reduce SEQ_LEN or check labeled row counts.')
    return (
        np.stack(sequences).astype('float32'),
        np.array(y, dtype='int64'),
        np.array(end_indices, dtype=np.int64),
    )

x, y, end_indices = build_sequences(df, encoded, labels, SEQ_LEN, STRIDE, MAX_SEQUENCES)
print('x:', x.shape, 'y:', y.shape)
print('sequence tensor memory MB:', round(x.nbytes / 1_000_000, 1))

# Free memory before training. test_rows keeps the rows needed for reporting.
import gc
keep_cols = [
    'timestamp', 'room_number', 'room_temp', 'setpoint', 'ideal_temp',
    'outside_temp', 'hvac_mode', 'room_state', LABEL_COL,
]
df = df[keep_cols].copy()
gc.collect()


x: (127154, 24, 42) y: (127154,)
sequence tensor memory MB: 512.7


0

In [17]:
# Chronological split by sequence end timestamp.
sequence_times = df.loc[end_indices, 'timestamp'].reset_index(drop=True)
order = np.argsort(sequence_times.to_numpy())
x = x[order]
y = y[order]
end_indices = end_indices[order]

n = len(x)
train_end = int(n * 0.80)
val_end = int(n * 0.90)
x_train, y_train = x[:train_end], y[:train_end]
x_val, y_val = x[train_end:val_end], y[train_end:val_end]
x_test, y_test = x[val_end:], y[val_end:]
test_indices = end_indices[val_end:]

print('train:', x_train.shape, 'val:', x_val.shape, 'test:', x_test.shape)
print('split times:', sequence_times.iloc[train_end], sequence_times.iloc[val_end])


train: (101723, 24, 42) val: (12715, 24, 42) test: (12716, 24, 42)
split times: 2022-01-25 07:20:00 2022-02-16 00:15:00


In [18]:
class_counts = np.bincount(y_train, minlength=len(classes)).astype('float32')
class_weights = class_counts.sum() / np.maximum(class_counts, 1.0)
class_weights = class_weights / class_weights.mean()
print('class_counts:', dict(zip(classes, class_counts.astype(int).tolist())))
print('class_weights:', dict(zip(classes, class_weights.round(3).tolist())))

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train)),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_val), torch.from_numpy(y_val)),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_test), torch.from_numpy(y_test)),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)


class_counts: {'AlwaysOnComfort': 30498, 'EnergySaver': 31648, 'Housekeeping': 1055, 'Preconditioning': 1657, 'Reactive': 36865}
class_weights: {'AlwaysOnComfort': 0.10000000149011612, 'EnergySaver': 0.09600000083446503, 'Housekeeping': 2.884999990463257, 'Preconditioning': 1.8370000123977661, 'Reactive': 0.08299999684095383}


In [19]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class TempreturePersonaTransformer(nn.Module):
    def __init__(self, input_dim, n_classes, d_model, n_heads, n_layers, dim_feedforward, dropout):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, d_model)
        self.position = PositionalEncoding(d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, n_classes),
        )

    def forward(self, x):
        hidden = self.input_projection(x)
        hidden = self.position(hidden)
        hidden = self.encoder(hidden)
        pooled = hidden.mean(dim=1)
        return self.head(pooled)

model = TempreturePersonaTransformer(
    input_dim=x_train.shape[-1],
    n_classes=len(classes),
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT,
).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=DEVICE))
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
print(model)


/tmp/ipykernel_5631/992374405.py:28: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)


TempreturePersonaTransformer(
  (input_projection): Linear(in_features=42, out_features=96, bias=True)
  (position): PositionalEncoding()
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=96, out_features=96, bias=True)
        )
        (linear1): Linear(in_features=96, out_features=192, bias=True)
        (dropout): Dropout(p=0.15, inplace=False)
        (linear2): Linear(in_features=192, out_features=96, bias=True)
        (norm1): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.15, inplace=False)
        (dropout2): Dropout(p=0.15, inplace=False)
      )
    )
  )
  (head): Sequential(
    (0): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=96, out_features=96, bias=True)
    (2): GELU(ap

In [20]:
def run_epoch(loader, train=False):
    model.train(train)
    total_loss = 0.0
    total_correct = 0
    total_count = 0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)
        if train:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(train):
            logits = model(xb)
            loss = criterion(logits, yb)
            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
        total_loss += loss.item() * len(xb)
        total_correct += (logits.argmax(dim=1) == yb).sum().item()
        total_count += len(xb)
    return total_loss / max(total_count, 1), total_correct / max(total_count, 1)

best_val = float('inf')
best_state = None
epochs_without_improvement = 0
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    scheduler.step(val_loss)
    history.append({'epoch': epoch, 'train_loss': train_loss, 'train_accuracy': train_acc, 'val_loss': val_loss, 'val_accuracy': val_acc})
    print(f'epoch {epoch:02d} train_loss={train_loss:.5f} train_acc={train_acc:.4f} val_loss={val_loss:.5f} val_acc={val_acc:.4f}')

    if val_loss < best_val - 1e-5:
        best_val = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print('early stopping')
            break

if best_state is not None:
    model.load_state_dict(best_state)


epoch 01 train_loss=0.61192 train_acc=0.5860 val_loss=0.53304 val_acc=0.6849
epoch 02 train_loss=0.34719 train_acc=0.7370 val_loss=0.54941 val_acc=0.6846
epoch 03 train_loss=0.27995 train_acc=0.7905 val_loss=0.53808 val_acc=0.7136
epoch 04 train_loss=0.21437 train_acc=0.8510 val_loss=0.52657 val_acc=0.7904
epoch 05 train_loss=0.15461 train_acc=0.8959 val_loss=0.47901 val_acc=0.8168
epoch 06 train_loss=0.13886 train_acc=0.9085 val_loss=0.47333 val_acc=0.8262
epoch 07 train_loss=0.12683 train_acc=0.9174 val_loss=0.50625 val_acc=0.8142
epoch 08 train_loss=0.11724 train_acc=0.9232 val_loss=0.42558 val_acc=0.8376
epoch 09 train_loss=0.10490 train_acc=0.9311 val_loss=0.40750 val_acc=0.8378
epoch 10 train_loss=0.10087 train_acc=0.9342 val_loss=0.36144 val_acc=0.8559
epoch 11 train_loss=0.09947 train_acc=0.9355 val_loss=0.31547 val_acc=0.8595
epoch 12 train_loss=0.08952 train_acc=0.9423 val_loss=0.31793 val_acc=0.8491
epoch 13 train_loss=0.08251 train_acc=0.9467 val_loss=0.33721 val_acc=0.8432

In [21]:
@torch.no_grad()
def predict(loader):
    model.eval()
    probs = []
    actual = []
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        logits = model(xb)
        probs.append(torch.softmax(logits, dim=1).detach().cpu().numpy())
        actual.append(yb.numpy())
    return np.concatenate(probs), np.concatenate(actual)

probs, actual = predict(test_loader)
pred = probs.argmax(axis=1)
accuracy = float((pred == actual).mean())
all_label_ids = list(range(len(classes)))
report = classification_report(
    actual,
    pred,
    labels=all_label_ids,
    target_names=classes,
    digits=4,
    zero_division=0,
)
cm = confusion_matrix(actual, pred, labels=all_label_ids)
print('accuracy:', accuracy)
print(report)


accuracy: 0.8103177099716892
                 precision    recall  f1-score   support

AlwaysOnComfort     0.9077    0.7537    0.8236      4150
    EnergySaver     0.9907    0.7158    0.8311      3870
   Housekeeping     0.9860    1.0000    0.9930       141
Preconditioning     0.9839    1.0000    0.9919       245
       Reactive     0.6610    0.9327    0.7737      4310

       accuracy                         0.8103     12716
      macro avg     0.9059    0.8804    0.8826     12716
   weighted avg     0.8517    0.8103    0.8141     12716



In [22]:
model_path = OUT_DIR / 'tempreture_persona_transformer.pt'
metadata_path = OUT_DIR / 'tempreture_persona_transformer_metadata.json'
report_path = OUT_DIR / 'tempreture_persona_transformer_report.txt'
cm_path = OUT_DIR / 'tempreture_persona_transformer_confusion_matrix.csv'
zip_path = OUT_DIR / 'tempreture_persona_transformer_outputs.zip'

metadata = {
    'model_type': 'transformer_classifier',
    'task': 'tempreture_persona_prediction',
    'label_col': LABEL_COL,
    'classes': classes,
    'class_to_id': class_to_id,
    'sequence_length': SEQ_LEN,
    'stride': STRIDE,
    'feature_names': feature_names,
    'base_numeric_cols': BASE_NUMERIC_COLS,
    'categorical_cols': CAT_COLS,
    'category_values': category_values,
    'input_dim': int(x_train.shape[-1]),
    'd_model': D_MODEL,
    'n_heads': N_HEADS,
    'n_layers': N_LAYERS,
    'dim_feedforward': DIM_FEEDFORWARD,
    'dropout': DROPOUT,
    'seed': SEED,
    'max_rows': MAX_ROWS,
    'max_sequences': MAX_SEQUENCES,
    'excluded_labels': sorted(EXCLUDED_LABELS),
    'history': history,
    'metrics': {'accuracy': accuracy},
}

torch.save({'state_dict': model.state_dict(), 'metadata': metadata}, model_path)
metadata_path.write_text(json.dumps(metadata, indent=2))

cm_df = pd.DataFrame(cm, index=classes, columns=classes)
cm_df.to_csv(cm_path)

full_report = '\n'.join([
    f'rows: {len(df):,}',
    f'sequences: {len(x):,}',
    f'train sequences: {len(x_train):,}',
    f'validation sequences: {len(x_val):,}',
    f'test sequences: {len(x_test):,}',
    f'sequence length: {SEQ_LEN}',
    f'accuracy: {accuracy:.4f}',
    '',
    report,
])
report_path.write_text(full_report)
print(full_report)


rows: 383,661
sequences: 127,154
train sequences: 101,723
validation sequences: 12,715
test sequences: 12,716
sequence length: 24
accuracy: 0.8103

                 precision    recall  f1-score   support

AlwaysOnComfort     0.9077    0.7537    0.8236      4150
    EnergySaver     0.9907    0.7158    0.8311      3870
   Housekeeping     0.9860    1.0000    0.9930       141
Preconditioning     0.9839    1.0000    0.9919       245
       Reactive     0.6610    0.9327    0.7737      4310

       accuracy                         0.8103     12716
      macro avg     0.9059    0.8804    0.8826     12716
   weighted avg     0.8517    0.8103    0.8141     12716



In [23]:
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in [model_path, metadata_path, report_path, cm_path]:
        zf.write(path, arcname=path.name)

print('saved model:', model_path)
print('saved metadata:', metadata_path)
print('saved report:', report_path)
print('saved confusion matrix:', cm_path)
print('created zip:', zip_path)
print('You can download the zip from Google Drive:', zip_path)


saved model: /content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/TempreturePersona/transformer/tempreture_persona_transformer.pt
saved metadata: /content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/TempreturePersona/transformer/tempreture_persona_transformer_metadata.json
saved report: /content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/TempreturePersona/transformer/tempreture_persona_transformer_report.txt
saved confusion matrix: /content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/TempreturePersona/transformer/tempreture_persona_transformer_confusion_matrix.csv
created zip: /content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/TempreturePersona/transformer/tempreture_persona_transformer_outputs.zip
You can download the zip from Google Drive: /content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/TempreturePersona/transformer/tempreture_persona_transformer_outputs.zip
